[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Linked Lists, Stacks, Queues, and Deques** {#linked-lists-stacks-queues-deques}

Arrays make position cheap because elements occupy contiguous slots. Linked structures make **local change** cheap because order is stored by references between nodes. Stacks, queues, and deques then place behavioral rules on how elements may enter and leave a collection.

These ideas are related, but they answer different design questions:

- a **linked list** chooses how a sequence is represented;
- a **stack**, **queue**, or **deque** specifies an ADT access policy;
- dummy nodes, fast/slow pointers, reversal, merging, and cycle detection are algorithms that manipulate or inspect links safely;
- a **monotonic stack** adds an invariant to an ordinary stack so that dominated candidates can be discarded permanently.

The most important linked-structure habit is to reason about references before values. Before changing <code>current.next</code>, identify which part of the structure would become unreachable and save the required reference. Before claiming an $O(1)$ insertion or deletion, state whether the target node or its predecessor has already been found; locating that position may still cost $O(n)$.


### **Linked List Fundamentals** {#linked-list-fundamentals}

A **linked list** represents an ordered sequence as individually allocated nodes. Each node stores a value and one or more references to neighboring nodes. A useful analogy is a treasure hunt: every clue contains both information and the location of the next clue. Unlike a row of numbered lockers, knowing the first location does not reveal the address of the fifth; the intermediate links must be followed.

Linked lists solve a different problem from arrays. They are useful when the sequence grows unpredictably, nodes must remain at stable locations, or insertions and deletions occur after already-known nodes. They are not automatically faster: indexed access and cache locality are worse because nodes may be scattered through memory.

![A singly linked list stores a value and a next reference in every node; the final reference is null.](assets/singly-linked-list.svg){fig-align="center" width="82%"}

*Open visual source: [Wikimedia Commons - Singly linked list](https://commons.wikimedia.org/wiki/File:Singly-linked-list.svg).*

Three common representations are:

- A **singly linked list** stores <code>next</code>. Traversal moves only forward.
- A **doubly linked list** stores <code>next</code> and <code>prev</code>. Given a node, removal and backward movement are easier, but every node uses another reference.
- A **circular list** links the final node back to the first node or to a sentinel. There is no null end link, so traversal requires an explicit stopping rule.

For a singly linked Sequence ADT with stored <code>head</code>, <code>tail</code>, and <code>size</code>, the operation contract is:

| ADT operation | Precondition | Required effect or result |
|---|---|---|
| <code>length()</code> | None | Return the number of data nodes. |
| <code>first()</code> | List is not empty | Return the head value without mutation. |
| <code>prepend(x)</code> | None | Add <code>x</code> before the old head. |
| <code>append(x)</code> | None | Add <code>x</code> after the old tail. |
| <code>insert_after(node, x)</code> | <code>node</code> belongs to the list | Link a new node immediately after <code>node</code>. |
| <code>pop_front()</code> | List is not empty | Remove and return the old head value. |
| <code>find(x)</code> | None | Return a matching node or a documented not-found result. |

The representation produces these costs for <code>n</code> nodes:

| Operation | Singly linked cost | Why |
|---|---:|---|
| <code>length</code>, <code>first</code> | $O(1)$ | Metadata and head are stored directly. |
| <code>prepend</code> | $O(1)$ | Only the new node and head reference change. |
| <code>append</code> | $O(1)$ with a tail; $O(n)$ without one | A stored tail avoids traversal. |
| <code>insert_after</code> | $O(1)$ after the node is known | Two next references are assigned. |
| <code>pop_front</code> | $O(1)$ | Head advances once. |
| <code>get(i)</code>, <code>find(x)</code> | $O(n)$ worst case | Links must be followed sequentially. |
| <code>pop_back</code> | $O(n)$ | A singly linked list must locate the tail's predecessor. |
| iterate all nodes | $\Theta(n)$ | Every node is visited once. |

The list occupies $\Theta(n)$ total space: one node per value plus one <code>next</code> reference per node. A doubly linked representation remains $\Theta(n)$ but has a larger constant because every node stores two links.

<details>
<summary>Python implementation: a singly linked Sequence ADT</summary>

~~~python
from dataclasses import dataclass
from typing import Generic, Iterator, TypeVar

T = TypeVar("T")


@dataclass
class Node(Generic[T]):
    value: T
    next: "Node[T] | None" = None


class SinglyLinkedList(Generic[T]):
    """A singly linked list with head, tail, and stored size."""

    def __init__(self) -> None:
        self._head: Node[T] | None = None
        self._tail: Node[T] | None = None
        self._size = 0

    def prepend(self, value: T) -> None:
        new_node = Node(value, self._head)
        self._head = new_node

        # The first inserted node is both head and tail.
        if self._tail is None:
            self._tail = new_node
        self._size += 1

    def append(self, value: T) -> None:
        new_node = Node(value)

        if self._tail is None:
            self._head = self._tail = new_node
        else:
            self._tail.next = new_node
            self._tail = new_node
        self._size += 1

    def pop_front(self) -> T:
        if self._head is None:
            raise IndexError("pop from an empty linked list")

        removed = self._head
        self._head = removed.next
        self._size -= 1

        # Removing the only node must also clear the tail.
        if self._head is None:
            self._tail = None
        return removed.value

    def find(self, target: T) -> Node[T] | None:
        current = self._head
        while current is not None:
            if current.value == target:
                return current
            current = current.next
        return None

    def __len__(self) -> int:
        return self._size

    def __iter__(self) -> Iterator[T]:
        current = self._head
        while current is not None:
            yield current.value
            current = current.next


values = SinglyLinkedList[int]()
values.append(2)
values.append(3)
values.prepend(1)
assert list(values) == [1, 2, 3]
assert values.find(2) is not None
assert values.pop_front() == 1
assert list(values) == [2, 3]
assert len(values) == 2
~~~

</details>

The list is preferable to an array when the program already holds the relevant node and performs frequent local link changes. If the dominant operation is <code>get(i)</code>, an array is usually the better representation.

**Practice.** [LeetCode 2 - Add Two Numbers](https://leetcode.com/problems/add-two-numbers/) exercises node-by-node traversal and construction without requiring random access.


### **Dummy Nodes and Pointer Manipulation** {#dummy-nodes-pointer-manipulation}

A **dummy node**, also called a sentinel, is a structural node that does not represent a normal data item. It is placed before the first data node, after the final node, or at both ends. Its purpose is to remove boundary-specific branches from pointer algorithms.

Without a dummy node, deleting the first data node changes <code>head</code>, while deleting any later node changes <code>previous.next</code>. A leading dummy turns both cases into the same operation: delete the node after <code>previous</code>. The dummy is useful not because it improves Big-O complexity, but because it makes the invariant and implementation uniform.

![A leading dummy node ensures that even the first data node has a predecessor.](assets/dummy-node-operations.svg){fig-align="center" width="92%"}

For removing every node whose value equals a target, maintain <code>previous</code> at the final node of the already-processed result. Inspect <code>previous.next</code> rather than advancing both pointers unconditionally.

**Pseudocode.**

~~~text
REMOVE-ALL(head, target)
    dummy.next <- head
    previous <- dummy

    while previous.next is not null
        if previous.next.value equals target
            previous.next <- previous.next.next
        else
            previous <- previous.next

    return dummy.next
~~~

When a node is removed, <code>previous</code> stays in place because its new successor has not yet been inspected. When a node is retained, <code>previous</code> advances. The invariant is: all nodes before <code>previous.next</code> have been processed, contain no target value, and preserve their original relative order.

<details>
<summary>Python implementation: remove values through a dummy predecessor</summary>

~~~python
from dataclasses import dataclass


@dataclass
class ListNode:
    value: int
    next: "ListNode | None" = None


def remove_all(head: ListNode | None, target: int) -> ListNode | None:
    """Remove all target-valued nodes while preserving the others."""
    dummy = ListNode(0, head)
    previous = dummy

    while previous.next is not None:
        if previous.next.value == target:
            # Bypass the target. Do not advance previous yet because
            # the new successor may also contain target.
            previous.next = previous.next.next
        else:
            previous = previous.next

    return dummy.next


def to_values(head: ListNode | None) -> list[int]:
    values: list[int] = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


head = ListNode(2, ListNode(2, ListNode(5, ListNode(2))))
assert to_values(remove_all(head, 2)) == [5]
assert remove_all(None, 2) is None
~~~

</details>

Each node is inspected once, so time is $O(n)$ and auxiliary space is $O(1)$. The dummy itself adds one constant-sized node. Dummy nodes are especially useful for deletion, partitioning, list merging, and constructing a result whose head is not known in advance.

**Practice.** [LeetCode 19 - Remove Nth Node From End of List](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) combines a dummy predecessor with a fixed gap between two pointers.


### **Fast and Slow Pointers** {#fast-slow-pointers}

The **fast and slow pointer pattern** traverses the same linked structure at different speeds. Slow usually moves one edge per round while fast moves two. The pattern replaces information that might otherwise require a length counter or visited-node set with a relationship between pointer positions.

For finding the middle of an acyclic list, every slow step corresponds to two fast steps. When fast has consumed the list, slow has consumed approximately half.

**Pseudocode.**

~~~text
MIDDLE(head)
    slow <- head
    fast <- head

    while fast is not null and fast.next is not null
        slow <- slow.next
        fast <- fast.next.next

    return slow
~~~

![Slow moves one node while fast moves two; when fast reaches null, slow is at the middle.](assets/fast-slow-pointers.svg){fig-align="center" width="94%"}

For a list of length <code>n</code>, the loop executes $\lfloor n/2 \rfloor$ times. On an even-length list, this initialization returns the **second** of the two middle nodes. Starting fast one node ahead or changing the loop condition produces a different convention, so pointer initialization is part of the algorithm's specification.

The technique is better than a two-pass “count then walk halfway” approach when one pass is required. Both use $O(1)$ auxiliary space. It also supports palindrome checking, splitting a list for merge sort, locating a fixed offset from the end, and cycle detection.

<details>
<summary>Python implementation: locate the second middle node</summary>

~~~python
from dataclasses import dataclass


@dataclass
class ListNode:
    value: int
    next: "ListNode | None" = None


def middle_node(head: ListNode | None) -> ListNode | None:
    """Return the middle, choosing the second middle for even length."""
    slow = head
    fast = head

    while fast is not None and fast.next is not None:
        slow = slow.next if slow is not None else None
        fast = fast.next.next

    return slow


odd = ListNode(1, ListNode(2, ListNode(3, ListNode(4, ListNode(5)))))
even = ListNode(1, ListNode(2, ListNode(3, ListNode(4))))
assert middle_node(odd).value == 3
assert middle_node(even).value == 3
assert middle_node(None) is None
~~~

</details>

Time is $O(n)$ because fast traverses at most <code>n</code> edges and slow at most half as many. Auxiliary space is $O(1)$ because only two references are stored.

**Practice.** [LeetCode 234 - Palindrome Linked List](https://leetcode.com/problems/palindrome-linked-list/) combines a fast/slow midpoint with reversing and comparing the second half.


### **Reversal, Merging, and Cycle Detection** {#reversal-merging-cycle-detection}

These algorithms are the core pointer transformations for singly linked lists. Reversal changes link direction, merging consumes two ordered fronts, and cycle detection reasons about repeated movement when a null terminator may never be reached.

#### **Reversing a Linked List** {#reversing-linked-list}

Reversal changes every <code>next</code> reference so the old tail becomes the new head. The main risk is losing the unprocessed suffix. Before redirecting <code>current.next</code>, save the old successor in <code>next_node</code>.

**Pseudocode.**

~~~text
REVERSE(head)
    previous <- null
    current <- head

    while current is not null
        next_node <- current.next
        current.next <- previous
        previous <- current
        current <- next_node

    return previous
~~~

![Linked-list reversal saves next, redirects the current link, and then advances previous and current.](assets/linked-list-reversal.svg){fig-align="center" width="92%"}

The invariant is: <code>previous</code> heads a correctly reversed prefix, while <code>current</code> heads the untouched suffix. Redirecting one link grows the reversed prefix by one node without losing the suffix because <code>next_node</code> was saved first.

<details>
<summary>Python implementation: in-place iterative reversal</summary>

~~~python
from dataclasses import dataclass


@dataclass
class ListNode:
    value: int
    next: "ListNode | None" = None


def reverse_list(head: ListNode | None) -> ListNode | None:
    previous: ListNode | None = None
    current = head

    while current is not None:
        # Save the suffix before redirecting current.next.
        next_node = current.next
        current.next = previous

        # Advance the boundary between reversed and unprocessed nodes.
        previous = current
        current = next_node

    return previous


head = ListNode(1, ListNode(2, ListNode(3)))
reversed_head = reverse_list(head)
assert reversed_head.value == 3
assert reversed_head.next.value == 2
assert reversed_head.next.next.value == 1
assert reversed_head.next.next.next is None
~~~

</details>

Every node is processed once, giving $O(n)$ time and $O(1)$ auxiliary space. A recursive reversal also takes $O(n)$ time but uses $O(n)$ call-stack space.

**Practice.** [LeetCode 206 - Reverse Linked List](https://leetcode.com/problems/reverse-linked-list/) directly tests this three-reference invariant.

#### **Merging Sorted Linked Lists** {#merging-sorted-linked-lists}

Merging exploits sorted order. Only the two current front nodes can be the next output node; every later node in either list is at least as large as its own front. A dummy result head avoids a special case for the first selected node, while <code>tail</code> marks the end of the already-merged prefix.

**Pseudocode.**

~~~text
MERGE-SORTED(left, right)
    dummy <- new node
    tail <- dummy

    while left is not null and right is not null
        if left.value <= right.value
            tail.next <- left
            left <- left.next
        else
            tail.next <- right
            right <- right.next
        tail <- tail.next

    tail.next <- whichever list remains
    return dummy.next
~~~

![The merge compares only the two current fronts and extends a sorted result through tail.](assets/merge-sorted-lists.svg){fig-align="center" width="92%"}

<details>
<summary>Python implementation: reuse nodes while merging</summary>

~~~python
from dataclasses import dataclass


@dataclass
class ListNode:
    value: int
    next: "ListNode | None" = None


def merge_sorted(
    left: ListNode | None,
    right: ListNode | None,
) -> ListNode | None:
    """Merge two ascending lists by relinking their existing nodes."""
    dummy = ListNode(0)
    tail = dummy

    while left is not None and right is not None:
        if left.value <= right.value:
            tail.next = left
            left = left.next
        else:
            tail.next = right
            right = right.next
        tail = tail.next

    # The remaining suffix is already sorted and can be linked at once.
    tail.next = left if left is not None else right
    return dummy.next


def values(head: ListNode | None) -> list[int]:
    result: list[int] = []
    while head is not None:
        result.append(head.value)
        head = head.next
    return result


left = ListNode(1, ListNode(4, ListNode(7)))
right = ListNode(2, ListNode(3, ListNode(8)))
assert values(merge_sorted(left, right)) == [1, 2, 3, 4, 7, 8]
~~~

</details>

If the input lengths are <code>n</code> and <code>m</code>, each node is linked once, so time is $O(n+m)$. Reusing input nodes requires $O(1)$ auxiliary space; constructing copies would require $O(n+m)$ additional node space.

**Practice.** [LeetCode 21 - Merge Two Sorted Lists](https://leetcode.com/problems/merge-two-sorted-lists/) applies this sorted-prefix invariant.

#### **Cycle Detection** {#linked-list-cycle-detection}

A cycle exists when following <code>next</code> eventually returns to an earlier node instead of reaching null. Remembering visited node identities detects repetition in $O(n)$ space. Floyd's two-pointer algorithm obtains the same answer with $O(1)$ space.

**Pseudocode.**

~~~text
HAS-CYCLE(head)
    slow <- head
    fast <- head

    while fast is not null and fast.next is not null
        slow <- slow.next
        fast <- fast.next.next
        if slow equals fast by node identity
            return true

    return false
~~~

![Once both pointers enter a cycle, fast gains one node per round and must eventually meet slow.](assets/cycle-detection.svg){fig-align="center" width="90%"}

Let $\mu$ be the number of edges before the cycle entry and $\lambda$ the cycle length. Once both pointers are inside the cycle, fast gains one node per round relative to slow. Their separation is measured modulo $\lambda$; repeatedly adding one must eventually produce separation zero, so they meet within at most $\lambda$ additional rounds.

To locate the cycle entry after a meeting, reset one pointer to <code>head</code> and move both one edge per round. The distance relationships at the meeting guarantee that they next coincide at the entry.

<details>
<summary>Python implementation: detect a cycle and return its entry</summary>

~~~python
from dataclasses import dataclass


@dataclass
class ListNode:
    value: int
    next: "ListNode | None" = None


def cycle_entry(head: ListNode | None) -> ListNode | None:
    """Return the cycle-entry node, or None for an acyclic list."""
    slow = head
    fast = head

    # Phase 1: find a meeting inside the cycle.
    while fast is not None and fast.next is not None:
        slow = slow.next if slow is not None else None
        fast = fast.next.next
        if slow is fast:
            break
    else:
        return None

    # Phase 2: equal-speed pointers meet at the cycle entry.
    seeker = head
    while seeker is not slow:
        seeker = seeker.next if seeker is not None else None
        slow = slow.next if slow is not None else None

    return seeker


first = ListNode(1)
entry = ListNode(2)
third = ListNode(3)
fourth = ListNode(4)
first.next = entry
entry.next = third
third.next = fourth
fourth.next = entry

assert cycle_entry(first) is entry
assert cycle_entry(ListNode(1, ListNode(2))) is None
~~~

</details>

Time is $O(n)$ and auxiliary space is $O(1)$. Identity comparison is essential: two different nodes may store equal values without forming a cycle.

**Practice.** [LeetCode 142 - Linked List Cycle II](https://leetcode.com/problems/linked-list-cycle-ii/) asks for the entry node rather than only a Boolean result.


### **Stacks and Queues** {#stacks-and-queues}

Stacks and queues are linear ADTs defined by **where removal is allowed**, not by one required storage layout. Both can use arrays or linked nodes; the ADT contract remains the same while operation costs depend on the representation.

#### **Stack ADT** {#stack-adt}

A **stack** follows last in, first out (LIFO) order, like trays added to and removed from the top of a pile. It is needed when the most recently suspended item must resume first: function calls, undo history, delimiter matching, depth-first search, and expression evaluation all have this shape.

![A stack allows insertion and removal only at the top.](assets/data-stack.svg){fig-align="center" width="42%"}

*Open visual source: [Wikimedia Commons - Data stack](https://commons.wikimedia.org/wiki/File:Data_stack.svg).*

| ADT operation | Precondition | Required effect or result |
|---|---|---|
| <code>push(x)</code> | None | Make <code>x</code> the new top. |
| <code>pop()</code> | Stack is not empty | Remove and return the top. |
| <code>peek()</code> | Stack is not empty | Return the top without removing it. |
| <code>is_empty()</code> | None | Report whether the stack has no items. |
| <code>size()</code> | None | Return the number of items. |

| Operation | Dynamic-array implementation | Linked implementation |
|---|---:|---:|
| <code>push</code> | amortized $O(1)$; worst $O(n)$ resize | worst-case $O(1)$ at head |
| <code>pop</code>, <code>peek</code> | $O(1)$ at array end | $O(1)$ at head |
| <code>is_empty</code>, <code>size</code> | $O(1)$ | $O(1)$ with stored size |
| total space | $\Theta(n)$ plus spare capacity | $\Theta(n)$ plus node links |

An array stack uses the final occupied index as the top so that pushing and popping avoid shifts. Using index 0 as the top of a Python list would make every operation $O(n)$ because the remaining elements shift.

<details>
<summary>Python implementation: an array-backed stack</summary>

~~~python
from typing import Generic, TypeVar

T = TypeVar("T")


class ArrayStack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if self.is_empty():
            raise IndexError("pop from an empty stack")
        return self._items.pop()

    def peek(self) -> T:
        if self.is_empty():
            raise IndexError("peek at an empty stack")
        return self._items[-1]

    def is_empty(self) -> bool:
        return len(self._items) == 0

    def __len__(self) -> int:
        return len(self._items)


stack = ArrayStack[str]()
stack.push("parse (")
stack.push("parse [")
assert stack.peek() == "parse ["
assert stack.pop() == "parse ["
assert stack.pop() == "parse ("
assert stack.is_empty()
~~~

</details>

**Practice.** [LeetCode 20 - Valid Parentheses](https://leetcode.com/problems/valid-parentheses/) uses a stack to ensure the most recently opened delimiter closes first.

#### **Queue ADT** {#queue-adt}

A **queue** follows first in, first out (FIFO) order, like customers served in arrival order. It is needed for scheduling, buffering, breadth-first search, event processing, and any workflow where older pending work must not be overtaken by newer work.

![A queue inserts at the rear and removes at the front, preserving arrival order.](assets/queue-operations.svg){fig-align="center" width="88%"}

| ADT operation | Precondition | Required effect or result |
|---|---|---|
| <code>enqueue(x)</code> | None | Add <code>x</code> at the rear. |
| <code>dequeue()</code> | Queue is not empty | Remove and return the front. |
| <code>front()</code> | Queue is not empty | Return the front without removal. |
| <code>is_empty()</code> | None | Report whether the queue has no items. |
| <code>size()</code> | None | Return the number of items. |

| Operation | Linked queue with front/rear | Circular-array queue |
|---|---:|---:|
| <code>enqueue</code> | $O(1)$ through rear | amortized $O(1)$ if resizable |
| <code>dequeue</code> | $O(1)$ through front | $O(1)$ by advancing front |
| <code>front</code>, <code>size</code> | $O(1)$ | $O(1)$ |
| total space | $\Theta(n)$ plus links | $\Theta(n)$ plus spare capacity |

Removing index 0 from a Python list is not a suitable queue implementation because it shifts every remaining element and costs $O(n)$. A linked queue stores both ends; a circular array advances indices instead of moving values.

<details>
<summary>Python implementation: a linked FIFO queue</summary>

~~~python
from dataclasses import dataclass
from typing import Generic, TypeVar

T = TypeVar("T")


@dataclass
class QueueNode(Generic[T]):
    value: T
    next: "QueueNode[T] | None" = None


class LinkedQueue(Generic[T]):
    def __init__(self) -> None:
        self._front: QueueNode[T] | None = None
        self._rear: QueueNode[T] | None = None
        self._size = 0

    def enqueue(self, value: T) -> None:
        new_node = QueueNode(value)

        if self._rear is None:
            self._front = self._rear = new_node
        else:
            self._rear.next = new_node
            self._rear = new_node
        self._size += 1

    def dequeue(self) -> T:
        if self._front is None:
            raise IndexError("dequeue from an empty queue")

        removed = self._front
        self._front = removed.next
        self._size -= 1

        # The empty queue must not retain a stale rear node.
        if self._front is None:
            self._rear = None
        return removed.value

    def front(self) -> T:
        if self._front is None:
            raise IndexError("front of an empty queue")
        return self._front.value

    def is_empty(self) -> bool:
        return self._size == 0

    def __len__(self) -> int:
        return self._size


queue = LinkedQueue[str]()
queue.enqueue("first")
queue.enqueue("second")
assert queue.front() == "first"
assert queue.dequeue() == "first"
assert queue.dequeue() == "second"
assert queue.is_empty()
~~~

</details>

**Practice.** [LeetCode 102 - Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/) uses a queue to process nodes in increasing distance from the root.


### **Deques** {#deques}

A **deque** (double-ended queue) allows insertion and removal at both ends. It combines the access policies of stacks and queues: one end can behave as a stack top, while opposite-end operations provide FIFO behavior. Deques are useful for work-stealing schedulers, undo/redo buffers, breadth-first search variants, and maintaining candidates inside a moving window.

| ADT operation | Precondition | Required effect or result |
|---|---|---|
| <code>append_left(x)</code> | None | Insert <code>x</code> as the new leftmost item. |
| <code>append_right(x)</code> | None | Insert <code>x</code> as the new rightmost item. |
| <code>pop_left()</code> | Deque is not empty | Remove and return the leftmost item. |
| <code>pop_right()</code> | Deque is not empty | Remove and return the rightmost item. |
| <code>peek_left()</code> / <code>peek_right()</code> | Deque is not empty | Observe an end without removing it. |
| <code>size()</code> | None | Return the number of items. |

A deque can use a doubly linked list with sentinel nodes or a circular array. A doubly linked list keeps direct references to both ends and can unlink an end node in constant time. A circular array stores <code>front</code>, <code>size</code>, and <code>capacity</code>; physical indices wrap with modulo arithmetic:

$$
\operatorname{physical}(i)
= (\operatorname{front}+i) \bmod \operatorname{capacity}.
$$

Here <code>i</code> is a logical position counted from the left. Modulo maps positions that pass the final array slot back to index 0, so no values shift when either end moves.

![A circular-array deque wraps indices around the storage block and changes either end without shifting elements.](assets/circular-deque.svg){fig-align="center" width="90%"}

| Operation | Circular array | Doubly linked list |
|---|---:|---:|
| append at either end | amortized $O(1)$; worst $O(n)$ resize | worst-case $O(1)$ |
| pop or peek at either end | $O(1)$ | $O(1)$ |
| indexed access | $O(1)$ if exposed | $O(n)$ |
| total space | $\Theta(n)$ plus spare capacity | $\Theta(n)$ plus two links per node |

The deque ADT does not normally promise efficient insertion in the middle. Choosing a deque over a list communicates that the program's important mutations occur at the ends.

<details>
<summary>Python implementation: a resizable circular-array deque</summary>

~~~python
from typing import Generic, TypeVar, cast

T = TypeVar("T")


class CircularDeque(Generic[T]):
    """A resizable deque backed by a circular array."""

    def __init__(self) -> None:
        self._data: list[T | None] = [None] * 4
        self._front = 0
        self._size = 0

    def _index(self, logical_index: int) -> int:
        return (self._front + logical_index) % len(self._data)

    def _grow(self) -> None:
        new_data: list[T | None] = [None] * (2 * len(self._data))

        # Copy in logical left-to-right order, then reset front to zero.
        for logical_index in range(self._size):
            new_data[logical_index] = self._data[self._index(logical_index)]

        self._data = new_data
        self._front = 0

    def append_left(self, value: T) -> None:
        if self._size == len(self._data):
            self._grow()

        self._front = (self._front - 1) % len(self._data)
        self._data[self._front] = value
        self._size += 1

    def append_right(self, value: T) -> None:
        if self._size == len(self._data):
            self._grow()

        self._data[self._index(self._size)] = value
        self._size += 1

    def pop_left(self) -> T:
        if self._size == 0:
            raise IndexError("pop from an empty deque")

        value = cast(T, self._data[self._front])
        self._data[self._front] = None
        self._front = (self._front + 1) % len(self._data)
        self._size -= 1
        return value

    def pop_right(self) -> T:
        if self._size == 0:
            raise IndexError("pop from an empty deque")

        right_index = self._index(self._size - 1)
        value = cast(T, self._data[right_index])
        self._data[right_index] = None
        self._size -= 1
        return value

    def peek_left(self) -> T:
        if self._size == 0:
            raise IndexError("peek at an empty deque")
        return cast(T, self._data[self._front])

    def peek_right(self) -> T:
        if self._size == 0:
            raise IndexError("peek at an empty deque")
        return cast(T, self._data[self._index(self._size - 1)])

    def __len__(self) -> int:
        return self._size


deque = CircularDeque[int]()
deque.append_right(2)
deque.append_right(3)
deque.append_left(1)
deque.append_right(4)
deque.append_right(5)  # triggers growth
assert deque.peek_left() == 1
assert deque.peek_right() == 5
assert deque.pop_left() == 1
assert deque.pop_right() == 5
assert len(deque) == 3
~~~

</details>

Python's <code>collections.deque</code> is the production choice for fast end operations. A plain list is suitable for stack behavior at its right end but not for repeated removal from its left end.

**Practice.** [LeetCode 239 - Sliding Window Maximum](https://leetcode.com/problems/sliding-window-maximum/) uses a monotonic deque to discard expired indices from the left and dominated values from the right.


### **Monotonic Stacks** {#monotonic-stacks}

A **monotonic stack** is an ordinary stack maintained in increasing or decreasing value order. The order is not the goal by itself; it represents unresolved candidates that have not yet encountered a value capable of answering their query.

For **next greater element**, scan left to right and store indices whose next greater value has not been found. The stack's values remain monotonically decreasing. When a larger current value arrives, it resolves every smaller index exposed at the top.

**Pseudocode.**

~~~text
NEXT-GREATER(values)
    answer <- array of -1 values
    stack <- empty stack of indices

    for i from 0 to length(values) - 1
        while stack is not empty and values[i] > values[stack.top]
            unresolved <- stack.pop()
            answer[unresolved] <- values[i]
        stack.push(i)

    return answer
~~~

![A decreasing stack keeps unresolved indices; the value 5 pops and resolves both 1 and 2.](assets/monotonic-stack-steps.svg){fig-align="center" width="94%"}

Two invariants explain the method:

1. indices in the stack are ordered by arrival, and their values are decreasing from bottom to top;
2. every index in the stack has no greater value between its position and the current scan position.

When the current value is greater than the top, it is the **first** greater value for that top index. Popping is permanent: a resolved index is never needed again. Storing indices rather than only values also preserves distance and position information.

<details>
<summary>Python implementation: next greater value with a decreasing stack</summary>

~~~python
def next_greater_values(values: list[int]) -> list[int]:
    """Return the next strictly greater value to the right, or -1."""
    answer = [-1] * len(values)
    unresolved: list[int] = []

    for index, value in enumerate(values):
        # Current value resolves every smaller candidate exposed on top.
        while unresolved and value > values[unresolved[-1]]:
            earlier_index = unresolved.pop()
            answer[earlier_index] = value

        # This index now waits for its own next greater value.
        unresolved.append(index)

    return answer


assert next_greater_values([2, 1, 5, 3]) == [5, 5, -1, -1]
assert next_greater_values([4, 3, 2, 1]) == [-1, -1, -1, -1]
~~~

</details>

Although the code contains a nested <code>while</code>, it is not quadratic. Each index is pushed once and popped at most once, so there are at most <code>2n</code> stack mutations. Total time is $O(n)$, auxiliary space is $O(n)$, and each individual iteration may still perform several pops. This is an amortized argument over the complete scan.

Increasing and decreasing stacks support next/previous greater/smaller queries, stock span, histogram area, rainwater boundaries, and subarray minimum/maximum contributions. The required comparison direction determines the monotonic invariant.

**Practice.** [LeetCode 739 - Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) stores unresolved day indices and uses their differences to report waiting times.


### **Comparison and Selection** {#comparison-selection}

Choose a representation from the operations the program performs repeatedly, not from the shape of one example.

| Structure or pattern | Central rule | Strong operations | Weak operations | Typical use |
|---|---|---|---|---|
| Singly linked list | each node points forward | known-node insertion, prepend, sequential traversal | random access, predecessor search | dynamic sequences and pointer algorithms |
| Doubly linked list | nodes point both ways | known-node removal, both-end mutation | indexing, per-node memory | LRU ordering and deque implementation |
| Stack | LIFO | push/pop/peek at one end | oldest-item access | undo, parsing, DFS |
| Queue | FIFO | enqueue rear, dequeue front | newest-item removal | scheduling, buffering, BFS |
| Deque | mutate both ends | four end operations | arbitrary middle mutation | sliding windows and work queues |
| Fast/slow pointers | unequal traversal speeds | midpoint and cycle reasoning | random access | linked-list structural inspection |
| Monotonic stack | ordered unresolved candidates | next/previous boundary queries | arbitrary search | nearest greater/smaller problems |

A practical selection sequence is:

1. identify whether the requirement is representation, access policy, or an algorithm over an existing structure;
2. state the ADT operations and their preconditions before selecting storage;
3. include the cost of **locating** a node, not only the cost of changing links after it is found;
4. use dummy nodes when the head or first result would otherwise need a separate branch;
5. write down pointer meanings and save reachability-critical references before mutation;
6. use amortized reasoning when each item can enter and leave a stack or deque only a bounded number of times.

Arrays remain preferable for frequent indexing and compact traversal. Linked nodes are preferable when stable node identity and local relinking dominate. Python applications should normally use built-in <code>list</code> for stacks and <code>collections.deque</code> for queues and deques; custom implementations are primarily for learning, specialized invariants, or systems-level control.

**Practice.** [LeetCode 146 - LRU Cache](https://leetcode.com/problems/lru-cache/) integrates a hash map with a doubly linked list to provide expected $O(1)$ lookup and recency updates.
